第一重：简单操作

In [2]:
# 导入相关需要的包
import math
import torch
import torch.nn as nn

import warnings
warnings.filterwarnings(action="ignore") # 忽略警告

# 设置随机种子
seed = 1234
torch.manual_seed(seed)



In [ ]:
# 单头自注意力
class SelfAttention(nn.Module):
    def __init__(self,hidden_dim,*args,**kwargs):
        super(SelfAttention,self).__init__()
        self.hidden_dim = hidden_dim

        self.q = nn.Linear(hidden_dim,hidden_dim)
        self.k = nn.Linear(hidden_dim,hidden_dim)
        self.v = nn.Linear(hidden_dim,hidden_dim)
    
    def forward(self,x):
        # 1.获取q,k,v
        q = self.q(x) # [B,L,D]
        k = self.k(x) # [B,L,D]
        v = self.v(x) # [B,L,D]

        # 2.计算注意力权重
        # k.T:[B,D,L] -> [D,L,B]
        attention_weights = (q @ k.transpose(-2,-1)) / math.sqrt(self.hidden_dim) # [B,L,L]

        #print(attention_weights)
        # 3.用softmax计算注意力权重
        # 固定一个query，对所有key加权求和为1
        # dim = 1 - 固定key，对所有query归一化
        # dim = -1 - 固定query，对所有key加权求和为1:第 i 个 key，应该把多少注意力分给各个 query？
        attention_weights = torch.softmax(attention_weights,dim=-1) # [B,L,L]
        #attention score 的每一行，是“一个 query 看所有 key 的打分”；softmax 应该让这一行变成概率分布，所以通常用 dim=-1


        # 4.加权求和
        output =  attention_weights @ v # [B,L,D]

        return output

X = torch.randn(3,4,2)
print(X)
net = SelfAttention(hidden_dim=2)
net(X)

        


tensor([[[ 1.4007e+00, -1.0216e+00],
         [-5.1132e-01, -5.1236e-01],
         [-6.5010e-01,  1.9048e+00],
         [-1.8408e+00,  9.6285e-01]],

        [[-1.0987e+00, -5.3510e-01],
         [-2.0040e-03,  1.0430e+00],
         [-2.6981e+00, -5.0815e-01],
         [ 8.7433e-01, -2.3583e-01]],

        [[ 5.9186e-01, -4.8238e-01],
         [ 8.9618e-02,  5.0087e-01],
         [ 1.3167e-01,  1.2114e-01],
         [-1.0315e+00, -7.1857e-01]]])


tensor([[[ 0.0815,  0.0468],
         [ 0.0434,  0.0522],
         [-0.0656,  0.1998],
         [-0.0329,  0.1271]],

        [[-0.1212, -0.3331],
         [-0.1059, -0.0111],
         [-0.1290, -0.4192],
         [-0.1082, -0.1719]],

        [[-0.2593,  0.1200],
         [-0.2539,  0.1517],
         [-0.2566,  0.1378],
         [-0.2692,  0.0864]]], grad_fn=<UnsafeViewBackward0>)

第二重：效率优化


In [16]:
class SelfAttention2(nn.Module):
    def __init__(self,hidden_dim,*args,**kwargs):
        super(SelfAttention2,self).__init__()

        self.hidden_dim = hidden_dim

        self.proj = nn.Linear(hidden_dim,hidden_dim*3)

        self.output_proj = nn.Linear(hidden_dim,hidden_dim)
    
    def forward(self,x):
        # 1.获取q,k,v
        qkv = self.proj(x)
        q,k,v = torch.split(qkv,self.hidden_dim,dim=-1)

        #2.计算注意力权重
        attention_weights = (q @ k.transpose(-2,-1)) / math.sqrt(self.hidden_dim)

        #3.用softmax计算注意力权重
        attention_weights = torch.softmax(attention_weights,dim=-1)

        # 4.加权求和
        output = attention_weights @ v

        return self.output_proj(output)

X = torch.rand(3, 2, 4)
net = SelfAttention2(4)
net(X).shape
        


torch.Size([3, 2, 4])

第三重：面试

In [ ]:
class SelfAttention3(nn.Module):
    def __init__(self,hidden_dim,*args,**kwargs):
        super(SelfAttention3,self).__init__()
        self.hidden_dim = hidden_dim

        self.query = nn.Linear(hidden_dim,hidden_dim)
        self.key = nn.Linear(hidden_dim,hidden_dim)
        self.value = nn.Linear(hidden_dim,hidden_dim)

        #一般是 0.1 的 dropout，一般写作 config.attention_probs_dropout_prob
        # hidden_dropout_prob 一般也是 0.1
        self.dropout = nn.Dropout(0.1)

        # 可以不写；具体和面试官沟通。
        # 这是 MultiHeadAttention 中的产物，这个留给 MultiHeadAttention 也没有问题；
        self.output_proj = nn.Linear(hidden_dim,hidden_dim)
    
    def forward(self,x,attention_mask = None):
        #1.得到q,k,v
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        #2.计算注意力权重
        attention_value = q @ k.transpose(-2,-1)
        attention_weights = attention_value / math.sqrt(self.hidden_dim)
        if attention_mask is not None:
            # 覆盖未来要预测的部分，用<pad>挡住
            attention_weights = attention_weights.masked_fill(
                attention_mask == 0,
                float("-1e20")
            )
        
        #3.算softmax
        attention_weights = torch.softmax(attention_weights,dim=-1)

        #4.dropout
        attention_weights = self.dropout(attention_weights)

        #5.output
        output = attention_weights @ v
        return self.output_proj(output)

X = torch.rand(3, 4, 2)
b = torch.tensor(
    [
        [1, 1, 1, 0],
        [1, 1, 0, 0],
        [1, 0, 0, 0],
    ]
)
print(b.shape)
mask = b.unsqueeze(dim=1).repeat(1, 4, 1)

net = SelfAttention3(2)
net(X, mask).shape


torch.Size([3, 4])


torch.Size([3, 4, 2])